# Scrapping the Indian Economic Association members list
**Note:** For reproducibility:



1.   First make a copy of the notebook and work on the copy
2.   Check these things: paths name, file names, APIs IDs, and customize them according to your environment
3.   After doing a major data operation, you can push your final version here: https://github.com/GaelKBertrand/scholar_info_append






### Mounting Google Drive as a working director

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Installing and loading all dependencies and libraries needed

In [2]:
#Install necessary dependencies and libraries
!pip install waybackpy
!pip install googlesearch-python
!pip install google-api-python-client
!pip install beautifulsoup4
!pip install fake_useragent #necessary for google search rapid automation
!pip install selenium
!apt-get update
!apt install -y chromium-chromedriver
!apt-get install -y wget
!pip install requests-html
!pip install flair
!pip install transformers
!pip install torch
!pip install sentence-transformers



import requests
from bs4 import BeautifulSoup
import pandas as pd
from time import sleep
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import logging


from waybackpy import WaybackMachineSaveAPI
from googlesearch import search
from googleapiclient.discovery import build
import time
import requests
from fake_useragent import UserAgent
from bs4 import BeautifulSoup
import logging

#from requests_html import HTMLSession
#from requests_html import AsyncHTMLSession
import pandas as pd
import logging
import asyncio
import nest_asyncio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.1/201.1 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 77.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 486.3/486.3 kB 32.3 MB/s eta 0:00:00
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,626 B]
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,646 kB]
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:10 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages

# Scrapping IEA members

## Extracting text

Initial check

In [ ]:
!pip install pdfminer.six
from pdfminer.high_level import extract_text

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 82.1 MB/s eta 0:00:00


In [ ]:
# Specify the path to your PDF in Google Drive
pdf_path = '/content/drive/My Drive/IEA/iea.pdf'   #this IEA is Indian Economics Association, not I-International

In [ ]:
text = extract_text(pdf_path)
print(text)


ANDRA PRADESH

AP-001  Adwant, Dr. Vijay Kumar 

AP-011  Basavaiah, Dr. C.

Opp. Nallakunta, P.O.O.U. Road, 
Hyderabad-500044,  
Andhra Pradesh.

Dept. of Economics SVU,  
P.G. Centre, Kavali-524201  
Andhra Pradesh.

AP-002  Ahmad Md. Yahiya 

AP-012  Bavaiah, Dr. M.D.

Mahabub Nagar-509001  
Andhra Pradesh.

AP-003  Ali, Dr. Mohammad Iqbal 

Subedari, Hanamkonda-506001, 
Andhra Pradesh.

AP-004  Ali, Dr. Shaik Mehar 

Ulavapadu-523292 Prakasam, 
Andhra Pradesh

AP-005  Apparao, Sri Settipalli 

Flat No. 302, Leela Grand,  
2nd Line, Devapuram, Guntur, 
Andhra Pradesh-522 002.

AP-006  Arika Dr. Someshwararao
C/o Sastry Banjaru Guda,  
Village-China Bagga, Post- Kotturu, 
Srikakulam - 532455.

AP-007  Babu, Dr. K. Madhu 

Department of Economics,  
Acharya Nagarjuna University, 
Nagarjuna Nagar, Guntur-522510, 
Andra Pradesh

AP-008  Babu, Kondaveeti Raja

SF-3, Soundarya, Apartment 
Punnamma Thota, M.G. Road, 
Vijayawada-520010, Dist. Krishna 
Andhra Pradesh

AP-009  Baig, Dr. Mirza 

Deep check

In [ ]:
from pdfminer.high_level import extract_pages
from pdfminer.layout import LTTextBoxHorizontal, LAParams
import io
import re

# Specify the path to your PDF
pdf_path = '/content/drive/My Drive/IEA/iea.pdf'

def extract_columns_from_pdf(pdf_path):
    laparams = LAParams()
    laparams.char_margin = 2.0
    laparams.line_margin = 0.5
    laparams.word_margin = 0.1

    def parse_lt_objs(lt_objs, columns):
        for lt_obj in lt_objs:
            if isinstance(lt_obj, LTTextBoxHorizontal):
                text = lt_obj.get_text()
                x0, x1 = lt_obj.bbox[0], lt_obj.bbox[2]
                # Assuming the columns are divided in the middle of the page width
                if x0 < median_width:
                    columns[0] += text
                else:
                    columns[1] += text
            elif hasattr(lt_obj, '_objs'):
                parse_lt_objs(lt_obj._objs, columns)

    columns = ['', '']
    for page_layout in extract_pages(pdf_path, laparams=laparams):
        # Assuming the columns are divided in the middle of the page width
        median_width = (page_layout.bbox[2] - page_layout.bbox[0]) / 2
        parse_lt_objs(page_layout, columns)
    return columns

def insert_markers(text):
    # Define the pattern for the code structures
    code_structures = [
        'AP-', 'AS-', 'BR-', 'CN-', 'CG-', 'DL-', 'GA-', 'GJ-', 'HP-', 'HR-', 'JK-', 'JH-',
        'KA-', 'KL-', 'MP-', 'MH-', 'ML-', 'MZ-', 'NL-', 'OD-', 'PY-', 'PB-', 'RJ-', 'SK-',
        'BR-', 'TN-', 'TL-', 'TR-', 'DD-', 'DN-', 'UP-', 'UK-', 'WB-'
    ]

    # Regular expression pattern to find all code IDs
    pattern = r'\b(' + '|'.join([re.escape(code) + r'\d{1,3}' for code in code_structures]) + r')\b'

    # Add 'begin-member' marker
    text_with_markers = re.sub(pattern, r'\nbegin-member \1', text)
    return '\n'.join(line.strip() for line in text_with_markers.splitlines())

# Extract text from the two-columns PDF
columns = extract_columns_from_pdf(pdf_path)

# Combine columns and insert markers
final_text = insert_markers('\n'.join(columns))
print(final_text)


ANDRA PRADESH

begin-member AP-001  Adwant, Dr. Vijay Kumar
Opp. Nallakunta, P.O.O.U. Road,
Hyderabad-500044,
Andhra Pradesh.

begin-member AP-002  Ahmad Md. Yahiya
Mahabub Nagar-509001
Andhra Pradesh.

begin-member AP-003  Ali, Dr. Mohammad Iqbal
Subedari, Hanamkonda-506001,
Andhra Pradesh.

begin-member AP-004  Ali, Dr. Shaik Mehar
Ulavapadu-523292 Prakasam,
Andhra Pradesh

begin-member AP-005  Apparao, Sri Settipalli
Flat No. 302, Leela Grand,
2nd Line, Devapuram, Guntur,
Andhra Pradesh-522 002.

begin-member AP-006  Arika Dr. Someshwararao
C/o Sastry Banjaru Guda,
Village-China Bagga, Post- Kotturu,
Srikakulam - 532455.

begin-member AP-007  Babu, Dr. K. Madhu
Department of Economics,
Acharya Nagarjuna University,
Nagarjuna Nagar, Guntur-522510,
Andra Pradesh

begin-member AP-008  Babu, Kondaveeti Raja
SF-3, Soundarya, Apartment
Punnamma Thota, M.G. Road,
Vijayawada-520010, Dist. Krishna
Andhra Pradesh

begin-member AP-009  Baig, Dr. Mirza Sarvar
Associate Professor,
H.No. 8-10, 28

Text extraction

In [ ]:
import re
import pandas as pd

# Use final_text variable instead of hardcoding text
text = final_text

def extract_info_into_csv(text):
    rows = []
    # Split text into pages using a placeholder "Page X" (assumes pages are marked in the text)
    pages = text.split("Page ")
    pages = [page.strip() for page in pages if page.strip()]  # Remove empty entries

    for page_number, page_content in enumerate(pages, start=1):
        print(f"Processing Page {page_number}...")  # Log page processing
        # Split the page into individual members
        members = page_content.split('begin-member ')[1:]

        for member in members:
            lines = member.strip().split('\n')
            code_line = lines[0].strip()
            if not code_line or ' ' not in code_line:
                continue

            # Extract the code and name, ensuring to swap first and last names and remove titles
            code, full_name = code_line.split(' ', 1)

            full_name = re.sub(r'Dr\. ?', '', full_name).strip()  # Remove "Dr" and any extra spaces
            name_parts = full_name.split(',')
            if len(name_parts) == 2:
                first_name = name_parts[0].strip()
                last_name = name_parts[1].strip()
            else:
                parts = full_name.split(' ', 1)
                first_name = parts[0].strip()
                last_name = ' '.join(parts[1:]).strip() if len(parts) > 1 else ''

            # Extract the address
            address_lines = []
            for line in lines[1:]:
                if 'begin-member' in line:
                    break
                address_lines.append(line.strip())

            address = ', '.join(address_lines).strip().replace(',,', ',')

            # Add page number to the extracted data for alignment
            rows.append([page_number, code, first_name, last_name, address])

    # Create DataFrame and save to CSV
    df = pd.DataFrame(rows, columns=['Page', 'Code', 'First Name', 'Last Name', 'Address'])
    print(df.head())  # Print the first few rows of the DataFrame
    df.to_csv('/content/drive/My Drive/IEA/iea_output.csv', index=False)
    print("Data successfully saved to CSV!")

# Extract information and save to CSV
extract_info_into_csv(text)


Processing Page 1...
   Page    Code First Name       Last Name  \
0     1  AP-001     Adwant     Vijay Kumar   
1     1  AP-002      Ahmad      Md. Yahiya   
2     1  AP-003        Ali  Mohammad Iqbal   
3     1  AP-004        Ali     Shaik Mehar   
4     1  AP-005    Apparao  Sri Settipalli   

                                             Address  
0  Opp. Nallakunta, P.O.O.U. Road, Hyderabad-5000...  
1              Mahabub Nagar-509001, Andhra Pradesh.  
2       Subedari, Hanamkonda-506001, Andhra Pradesh.  
3          Ulavapadu-523292 Prakasam, Andhra Pradesh  
4  Flat No. 302, Leela Grand, 2nd Line, Devapuram...  
Data successfully saved to CSV!


In [ ]:
import re
import pandas as pd

# Use final_text variable instead of hardcoding text
text = final_text

def extract_info_into_csv(text):
    rows = []
    members = text.split('begin-member ')[1:]

    for member in members:
        lines = member.strip().split('\n')
        code_line = lines[0].strip()
        if not code_line or ' ' not in code_line:
            continue

        # Extract the code and name, ensuring to swap first and last names and remove titles
        code, full_name = code_line.split(' ', 1)

        full_name = re.sub(r'Dr\. ?', '', full_name).strip()  # Remove "Dr" and any extra spaces
        name_parts = full_name.split(',')
        if len(name_parts) == 2:
            first_name = name_parts[0].strip()
            last_name = name_parts[1].strip()
        else:
            parts = full_name.split(' ', 1)
            first_name = parts[0].strip()
            last_name = ' '.join(parts[1:]).strip() if len(parts) > 1 else ''

        # Extract the address
        address_lines = []
        for line in lines[1:]:
            if 'begin-member' in line:
                break
            address_lines.append(line.strip())

        address = ', '.join(address_lines).strip().replace(',,', ',')
        rows.append([code, first_name, last_name, address])

    # Create DataFrame and save to CSV
    df = pd.DataFrame(rows, columns=['Code', 'First Name', 'Last Name', 'Address'])
    print(df.head())  # Print the first few rows of the DataFrame
    df.to_csv('/content/drive/My Drive/IEA/iea_output.csv', index=False)
    print("Data successfully saved to CSV!")

# Extract information and save to CSV
extract_info_into_csv(text)


     Code First Name       Last Name  \
0  AP-001     Adwant     Vijay Kumar   
1  AP-002      Ahmad      Md. Yahiya   
2  AP-003        Ali  Mohammad Iqbal   
3  AP-004        Ali     Shaik Mehar   
4  AP-005    Apparao  Sri Settipalli   

                                             Address  
0  Opp. Nallakunta, P.O.O.U. Road, Hyderabad-5000...  
1              Mahabub Nagar-509001, Andhra Pradesh.  
2       Subedari, Hanamkonda-506001, Andhra Pradesh.  
3          Ulavapadu-523292 Prakasam, Andhra Pradesh  
4  Flat No. 302, Leela Grand, 2nd Line, Devapuram...  
Data successfully saved to CSV!


In [ ]:
import pandas as pd
import re
from flair.models import SequenceTagger
from flair.data import Sentence

# Load the NER tagger
tagger = SequenceTagger.load("flair/ner-english")

# Updated keyword lists
profession_keywords_variety = [
    'professor', 'Professor', 'PROF', 'Prof.', 'assistant', 'Assistant', 'Asst.', 'ASSISTANT', 'Asst',
    'researcher', 'Researcher', 'RESEARCHER', 'research', 'Research', 'RESEARCH', 'head', 'Head', 'HEAD',
    'scholar', 'Scholar', 'SCHOLAR',
    'principal', 'Principal', 'PRINCIPAL', 'lecturer', 'Lecturer', 'LECTURER', 'faculty', 'Faculty', 'FACULTY',
    'chief general manager', 'Chief General Manager', 'CGM', 'CGM',
    'faculty member', 'Faculty Member', 'FACULTY MEMBER', 'research scholar', 'Research Scholar', 'RESEARCH SCHOLAR',
    'reader in economics', 'Reader in Economics', 'READER IN ECONOMICS'
]
institution_keywords_variety = [
    'university', 'University', 'Univ.', 'UNIV', 'college', 'College', 'COLL', 'COLLEGE', 'institute', 'Institute',
    'INSTITUTE', 'inst', 'Inst', 'INST', 'department', 'Department', 'DEPT', 'DEPARTMENT', 'dept', 'Dept', 'DEPT',
    'school', 'School', 'SCH', 'SCHOOL', 'academy', 'Academy', 'ACAD', 'ACADEMY',
    'national institute', 'National Institute', 'NATIONAL INSTITUTE', 'St. Joseph College', 'ST. JOSEPH COLLEGE', 'Bank', 'Reserve Bank', 'Campus'
]

# Updated function to check if text contains any keyword
def contains_any_keyword(text, keywords):
    text_lower = text.lower()
    return any(keyword.lower() in text_lower for keyword in keywords)

# Function to extract and classify information into a DataFrame
def extract_info_into_csv(text):
    rows = []
    members = text.split('begin-member ')[1:]

    for member in members:
        lines = member.strip().split('\n')
        code_line = lines[0].strip()
        if not code_line or ' ' not in code_line:
            continue

        # Extract code and name, ensuring to swap first and last names and remove titles, keeping post-name text separate
        code, full_name = code_line.split(' ', 1)
        full_name = re.sub(r'(Dr\.|Prof\.) ?', '', full_name).strip()
        name_parts = full_name.split(',')
        if len(name_parts) == 2:
            last_name = name_parts[0].strip()
            first_name = name_parts[1].strip()
        else:
            parts = full_name.split(' ', 1)
            first_name = parts[0].strip()
            last_name = ' '.join(parts[1:]).strip() if len(parts) > 1 else ''

        # Extract remaining text after collecting names
        remaining_text = ' '.join(lines[1:])

        # Initialize variables
        profession, institution = '', ''
        address_lines = []

        # Split remaining text by comma and process each part
        for part in remaining_text.split(','):
            part = part.strip()
            if not profession and contains_any_keyword(part, profession_keywords_variety):
                profession = part
            elif not institution and contains_any_keyword(part, institution_keywords_variety):
                institution = part
            else:
                address_lines.append(part)

        # Join address parts and clean up extra commas
        address = ', '.join(address_lines).replace(',,', ',').replace(',.', '').strip()
        sentence = Sentence(address)
        tagger.predict(sentence)

        revised_address_parts = []

        for entity in sentence.get_spans('ner'):
            entity_text = entity.text.strip(',')
            if not contains_any_keyword(entity_text, profession_keywords_variety + institution_keywords_variety):
                revised_address_parts.append(entity_text.strip(','))

        refined_address = ', '.join(revised_address_parts).strip()

        # Clean up further commas
        refined_address = re.sub(r'(\s*,\s*)+', ', ', refined_address).strip()

        # Print the results for validation
        print(f"Code: {code}\nFirst Name: {first_name}\nLast Name: {last_name}\nProfession: {profession}\nInstitution: {institution}\nAddress: {refined_address}\n")

        rows.append([code, first_name, last_name, profession, institution, refined_address])

    # Create DataFrame and save to CSV
    df = pd.DataFrame(rows, columns=['Code', 'First Name', 'Last Name', 'Profession', 'Institution', 'Address'])
    print(df.head()) # Show first few rows of the DataFrame
    df.to_csv('/content/drive/My Drive/IEA/iea_output.csv', index=False)
    print("Data successfully saved to CSV!")

# Extract information and save to CSV
extract_info_into_csv(text)


2025-01-21 11:41:50,161 SequenceTagger predicts: Dictionary with 20 tags: <unk>, O, S-ORG, S-MISC, B-PER, E-PER, S-LOC, B-ORG, E-ORG, I-PER, S-PER, B-MISC, I-MISC, E-MISC, I-ORG, B-LOC, E-LOC, I-LOC, <START>, <STOP>
Processing member 1 of 6565...


AttributeError: 're.Pattern' object has no attribute 'union'

In [ ]:
import pandas as pd
import re
from flair.models import SequenceTagger
from flair.data import Sentence

# Load the NER tagger
tagger = SequenceTagger.load("flair/ner-english")

# Updated keyword lists
profession_keywords_variety = [
    'professor', 'Professor', 'PROF', 'Prof.', 'assistant', 'Assistant', 'Asst.', 'ASSISTANT', 'Asst',
    'researcher', 'Researcher', 'RESEARCHER', 'research', 'Research', 'RESEARCH', 'head', 'Head', 'HEAD',
    'scholar', 'Scholar', 'SCHOLAR',
    'principal', 'Principal', 'PRINCIPAL', 'lecturer', 'Lecturer', 'LECTURER', 'faculty', 'Faculty', 'FACULTY',
    'chief general manager', 'Chief General Manager', 'CGM', 'CGM',
    'faculty member', 'Faculty Member', 'FACULTY MEMBER', 'research scholar', 'Research Scholar', 'RESEARCH SCHOLAR',
    'reader in economics', 'Reader in Economics', 'READER IN ECONOMICS'
]
institution_keywords_variety = [
    'university', 'University', 'Univ.', 'UNIV', 'college', 'College', 'COLL', 'COLLEGE', 'institute', 'Institute',
    'INSTITUTE', 'inst', 'Inst', 'INST', 'department', 'Department', 'DEPT', 'DEPARTMENT', 'dept', 'Dept', 'DEPT',
    'school', 'School', 'SCH', 'SCHOOL', 'academy', 'Academy', 'ACAD', 'ACADEMY',
    'national institute', 'National Institute', 'NATIONAL INSTITUTE', 'St. Joseph College', 'ST. JOSEPH COLLEGE', 'Bank', 'Reserve Bank', 'Campus'
]

# Combine the keywords into sets for faster lookup
profession_keywords_set = set([keyword.lower() for keyword in profession_keywords_variety])
institution_keywords_set = set([keyword.lower() for keyword in institution_keywords_variety])

# Updated function to check if text contains any keyword (now using set for O(1) complexity)
def contains_any_keyword(text, keywords_set):
    text_lower = text.lower()
    return any(keyword in text_lower for keyword in keywords_set)

# Function to extract and classify information into a DataFrame
def extract_info_into_csv(text):
    rows = []
    members = text.split('begin-member ')[1:]

    for member in members:
        lines = member.strip().split('\n')
        code_line = lines[0].strip()
        if not code_line or ' ' not in code_line:
            continue

        # Extract code and name, ensuring to swap first and last names and remove titles, keeping post-name text separate
        code, full_name = code_line.split(' ', 1)
        full_name = re.sub(r'(Dr\.|Prof\.) ?', '', full_name).strip()
        name_parts = full_name.split(',')
        if len(name_parts) == 2:
            first_name = name_parts[0].strip()  # First name should be after the comma
            last_name = name_parts[1].strip()   # Last name should be before the comma
        else:
            parts = full_name.split(' ', 1)
            first_name = parts[0].strip()
            last_name = ' '.join(parts[1:]).strip() if len(parts) > 1 else ''

        # Extract remaining text after collecting names
        remaining_text = ' '.join(lines[1:])

        # Initialize variables
        profession, institution = '', ''
        address_lines = []

        # Split remaining text by comma and process each part
        for part in remaining_text.split(','):
            part = part.strip()
            if not profession and contains_any_keyword(part, profession_keywords_set):
                profession = part
            elif not institution and contains_any_keyword(part, institution_keywords_set):
                institution = part
            else:
                address_lines.append(part)

        # Join address parts and clean up extra commas
        address = ', '.join(address_lines).replace(',,', ',').replace(',.', '').strip()
        sentence = Sentence(address)
        tagger.predict(sentence)

        revised_address_parts = []

        for entity in sentence.get_spans('ner'):
            entity_text = entity.text.strip(',')
            if not contains_any_keyword(entity_text, profession_keywords_set.union(institution_keywords_set)):
                revised_address_parts.append(entity_text.strip(','))

        refined_address = ', '.join(revised_address_parts).strip()

        # Clean up further commas
        refined_address = re.sub(r'(\s*,\s*)+', ', ', refined_address).strip()

        # Print the results for validation
        print(f"Code: {code}\nFirst Name: {first_name}\nLast Name: {last_name}\nProfession: {profession}\nInstitution: {institution}\nAddress: {refined_address}\n")

        rows.append([code, first_name, last_name, profession, institution, refined_address])

    # Create DataFrame and save to CSV
    df = pd.DataFrame(rows, columns=['Code', 'First Name', 'Last Name', 'Profession', 'Institution', 'Address'])
    print(df.head()) # Show first few rows of the DataFrame
    df.to_csv('/content/drive/My Drive/IEA/iea_output.csv', index=False)
    print("Data successfully saved to CSV!")

# Extract information and save to CSV
extract_info_into_csv(text)


pytorch_model.bin:   3%|2         | 10.5M/419M [00:00<?, ?B/s]

Streaming output truncated to the last 5000 lines.
Last Name: G.
Profession: Assistant Professor
Institution: Department of Commerce
Address: Pallavaram, Chennai-600117

Code: TN-796
First Name: Krishnan
Last Name: A.
Profession: Associate Professor
Institution: Department of Commerce
Address: Pallavaram, Chennai-600117

Code: TN-803
First Name: Rajmohan
Last Name: Mr. Kaushik U
Profession: 
Institution: 
Address: Chennai, Second Main Road, Anna Nagar East, Chennai-600102

Code: TN-804
First Name: Sunitha
Last Name: C.
Profession: Assistant Professor
Institution: Department of Economics
Address: Chennai

Code: TN-805
First Name: K
Last Name: Nandini
Profession: Assistant Professor
Institution: Department of Economics
Address: Chennai-600008

Code: TN-806
First Name: A
Last Name: Saradha
Profession: Assistant Professor
Institution: Department of Economics
Address: Chennai-600008

Code: TN-807
First Name: Selvamani
Last Name: 
Profession: Assistant Professor
Institution: Department of Bu

In [ ]:
import pandas as pd

# Load the output CSV file
file_path = '/content/drive/My Drive/IEA/iea_output.csv'
df = pd.read_csv(file_path)

# Function to remove prefixes from names
def remove_prefixes(name):
    if isinstance(name, str):  # Only process if the value is a string
        prefixes = ['Mr.', 'Mrs.', 'Ms.', 'Mister']
        for prefix in prefixes:
            if name.startswith(prefix):  # Check if the text starts with the prefix
                name = name[len(prefix):].strip()  # Remove the prefix and strip extra spaces
        return name
    return name  # Return the value as is if it's not a string

# Apply the function to both 'First Name' and 'Last Name' columns
df['First Name'] = df['First Name'].apply(remove_prefixes)
df['Last Name'] = df['Last Name'].apply(remove_prefixes)

# Save the cleaned DataFrame back to a new CSV file
output_path = '/content/drive/My Drive/IEA/iea_output_cleaned.csv'
df.to_csv(output_path, index=False)

print("Prefixes removed and data saved to:", output_path)


Prefixes removed and data saved to: /content/drive/My Drive/IEA/iea_output_cleaned.csv


In [ ]:
import pandas as pd

# Load the cleaned CSV file
file_path = '/content/drive/My Drive/IEA/iea_output_cleaned.csv'
df = pd.read_csv(file_path)

# Define prefixes to check
prefixes = ['Mr.', 'Mrs.', 'Ms.', 'Mister']

# Function to check if a prefix exists in a string
def check_prefix(name):
    if isinstance(name, str):  # Ensure the value is a string
        for prefix in prefixes:
            if name.startswith(prefix):
                return True  # Prefix found
    return False  # No prefix or not a string

# Check for prefixes in both 'First Name' and 'Last Name' columns
df['Prefix in First Name'] = df['First Name'].apply(check_prefix)
df['Prefix in Last Name'] = df['Last Name'].apply(check_prefix)

# Count rows where prefixes still exist
first_name_count = df['Prefix in First Name'].sum()
last_name_count = df['Prefix in Last Name'].sum()

print(f"Prefixes found in 'First Name': {first_name_count}")
print(f"Prefixes found in 'Last Name': {last_name_count}")


Prefixes found in 'First Name': 0
Prefixes found in 'Last Name': 0


In [ ]:
!pip install transformers torch

import pandas as pd
from transformers import pipeline

# Load the Zero-Shot Classification Pipeline
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# Define labels for gender
labels = ["Male", "Female"]

# Function to predict gender
def predict_gender(first_name, last_name):
    full_name = f"{first_name} {last_name}".strip()
    if not full_name:
        return "Unknown"  # Handle missing names

    # Classify the name
    result = classifier(full_name, candidate_labels=labels, multi_label=False)
    return result["labels"][0]  # Return the most likely label

# Load your dataset
file_path = "/content/drive/My Drive/IEA/iea_output_cleaned.csv"  # Update with your file path
df = pd.read_csv(file_path)

# Apply the gender prediction function with progress tracking
def process_row(row):
    code = row["Code"]
    first_name = row["First Name"]
    last_name = row["Last Name"]

    # Predict gender
    gender = predict_gender(first_name, last_name)

    # Print progress
    print(f"Processing Code: {code}, First Name: {first_name}, Last Name: {last_name}, Predicted Gender: {gender}")

    return gender

df["Gender"] = df.apply(process_row, axis=1)

# Save the updated DataFrame with gender column
output_file_path = "/content/drive/My Drive/IEA/iea_output_with_gender.csv"
df.to_csv(output_file_path, index=False)

print(f"Updated file saved to {output_file_path}")


Device set to use cpu


Streaming output truncated to the last 5000 lines.
Processing Code: MP-247, First Name: Jain, Last Name: Anshul, Predicted Gender: Male
Processing Code: MP-248, First Name: Jaiswal, Last Name: Aslendra Kumar, Predicted Gender: Male
Processing Code: MP-248, First Name: Dhanora, Last Name: Madan, Predicted Gender: Female
Processing Code: MP-249, First Name: Sen, Last Name: Aneeta, Predicted Gender: Female
Processing Code: MH-001, First Name: Abhyankar, Last Name: Aditi Avinash, Predicted Gender: Female
Processing Code: MH-002, First Name: Acharya, Last Name: Deepa D., Predicted Gender: Female
Processing Code: MH-003, First Name: Acharya, Last Name: Hemlata, Predicted Gender: Female
Processing Code: MH-004, First Name: Achyutrao, Last Name: Rasal Rajendra, Predicted Gender: Male
Processing Code: MH-005, First Name: Ade, Last Name: Nitin Dhanusing, Predicted Gender: Male
Processing Code: MH-006, First Name: Adkime, Last Name: Baliram Shivanand, Predicted Gender: Female
Processing Code: MH-

In [ ]:
import pandas as pd

# Load the updated file with Gender column
file_path = "/content/drive/My Drive/IEA/iea_output_with_gender.csv"  # Update with your file path
df = pd.read_csv(file_path)

# Add the gender_encode column
df["gender_encode"] = df["Gender"].apply(lambda x: 1 if x == "Female" else 0)

# Save the updated DataFrame
output_file_path = "/content/drive/My Drive/IEA/iea_output_with_gender.csv"
df.to_csv(output_file_path, index=False)

print(f"Updated file with 'gender_encode' saved to {output_file_path}")


Updated file with 'gender_encode' saved to /content/drive/My Drive/IEA/iea_output_with_gender.csv


In [ ]:
import pandas as pd
import re

# Define the code structures
code_structures = [
    'AP-', 'AS-', 'BR-', 'CN-', 'CG-', 'DL-', 'GA-', 'GJ-', 'HP-', 'HR-', 'JK-', 'JH-',
    'KA-', 'KL-', 'MP-', 'MH-', 'ML-', 'MZ-', 'NL-', 'OD-', 'PY-', 'PB-', 'RJ-', 'SK-',
    'BR-', 'TN-', 'TL-', 'TR-', 'DD-', 'DN-', 'UP-', 'UK-', 'WB-'
]

# Regular expression pattern for sorting logic
pattern = r'^(' + '|'.join(map(re.escape, code_structures)) + r')(\d{1,3})$'

# Load the processed file
file_path = '/content/drive/My Drive/IEA/iea_output_with_gender.csv'
df = pd.read_csv(file_path)

# Sorting logic: Extract prefix and numeric part for proper sorting
def sort_key(code):
    match = re.match(pattern, code)
    if match:
        prefix, number = match.groups()
        return prefix, int(number)
    return code, float('inf')  # Place invalid codes at the end

# Sort the DataFrame based on the 'Code' column
df = df.sort_values(by='Code', key=lambda col: col.map(sort_key))

# Save the sorted DataFrame back to a new file
sorted_file_path = '/content/drive/My Drive/IEA/iea_output_sorted.csv'
df.to_csv(sorted_file_path, index=False)

print("Sorting completed and saved to:", sorted_file_path)


Sorting completed and saved to: /content/drive/My Drive/IEA/iea_output_sorted.csv


In [3]:
import pandas as pd
import re

# Define the code structures
code_structures = [
    'AP-', 'AS-', 'BR-', 'CN-', 'CG-', 'DL-', 'GA-', 'GJ-', 'HP-', 'HR-', 'JK-', 'JH-',
    'KA-', 'KL-', 'MP-', 'MH-', 'ML-', 'MZ-', 'NL-', 'OD-', 'PY-', 'PB-', 'RJ-', 'SK-',
    'BR-', 'TN-', 'TL-', 'TR-', 'DD-', 'DN-', 'UP-', 'UK-', 'WB-'
]

# Regular expression to extract the prefix and numeric part
pattern = r'^(' + '|'.join(map(re.escape, code_structures)) + r')(\d{1,4})$'

# Load the processed file
file_path = '/content/drive/My Drive/IEA/iea_output_sorted_test.csv'
df = pd.read_csv(file_path)

# Function to check consecutive numbers for each prefix
def check_consecutive_numbers(numbers):
    numbers = sorted(numbers)  # Sort the numbers
    missing_numbers = []
    for i in range(len(numbers) - 1):
        if numbers[i + 1] != numbers[i] + 1:
            missing_numbers.extend(range(numbers[i] + 1, numbers[i + 1]))
    return missing_numbers

# Test each prefix for missing numbers
for prefix in code_structures:
    # Filter codes matching the current prefix
    prefix_df = df[df['Code'].str.startswith(prefix, na=False)]

    # Extract numeric parts
    prefix_numbers = prefix_df['Code'].str.extract(pattern)
    prefix_numbers = prefix_numbers.dropna()
    prefix_numbers[1] = prefix_numbers[1].astype(int)  # Convert numeric part to integer

    # Check for missing numbers
    missing = check_consecutive_numbers(prefix_numbers[1].tolist())

    # Print results
    print(f"Prefix: {prefix}")
    print(f"  Total Codes: {len(prefix_numbers)}")
    print(f"  Missing Numbers: {missing if missing else 'None'}")
    print("-" * 50)


Prefix: AP-
  Total Codes: 304
  Missing Numbers: [117, 192, 201]
--------------------------------------------------
Prefix: AS-
  Total Codes: 113
  Missing Numbers: [35, 42, 72]
--------------------------------------------------
Prefix: BR-
  Total Codes: 842
  Missing Numbers: [91, 343, 666, 671, 672, 680, 681, 683, 684, 685, 686, 688, 694, 695, 738, 752, 755]
--------------------------------------------------
Prefix: CN-
  Total Codes: 67
  Missing Numbers: [21]
--------------------------------------------------
Prefix: CG-
  Total Codes: 82
  Missing Numbers: None
--------------------------------------------------
Prefix: DL-
  Total Codes: 301
  Missing Numbers: [30, 34, 42, 50, 123, 201, 265]
--------------------------------------------------
Prefix: GA-
  Total Codes: 39
  Missing Numbers: None
--------------------------------------------------
Prefix: GJ-
  Total Codes: 185
  Missing Numbers: [41, 95, 118]
--------------------------------------------------
Prefix: HP-
  Total 

Clean names

In [6]:
# Import necessary libraries
import pandas as pd

# Define the file path
file_path = '/content/drive/My Drive/IEA/iea_output_sorted_test.csv'

# Load the dataset
data = pd.read_csv(file_path)

# List of common Indian surnames (identified earlier)
common_surnames = [
    'Acharya', 'Agarwal', 'Agrawal', 'Banerjee', 'Barman', 'Bhattacharjee', 'Bhattacharya',
    'Biswas', 'Chakraborty', 'Chatterjee', 'Choudhary', 'Das', 'Dasgupta', 'Dutta', 'Ghosh',
    'Gupta', 'Hazra', 'Jha', 'Kapoor', 'Karmakar', 'Mandal', 'Mazumdar', 'Mukherjee',
    'Naik', 'Nair', 'Pandey', 'Patel', 'Rao', 'Roy', 'Sahu', 'Sengupta', 'Sen', 'Sharma',
    'Singh', 'Srivastava', 'Talukdar', 'Tripathi', 'Upadhyay', 'Varma', 'Verma', 'Yadav', 'Kumar', 'Prasad', 'Priyadarshi', 'Rai', 'Raj',
    'Ram', 'Sah', 'Tiwari', 'Thakur', 'Kumari', 'Khanna', 'Chadha', 'Kapur', 'Kaur', 'Mishra', 'Khanna', 'Reddy', 'Devi', 'Dey', 'Bhagat', 'Chaudhary', 'Chauhan',
    '',
]

# Function to reverse names if a surname is found in the wrong column
def correct_name_order(row):
    if row['First Name'] in common_surnames:
        # Swap the columns if the first name is a common surname
        return pd.Series([row['Last Name'], row['First Name']])
    return pd.Series([row['First Name'], row['Last Name']])

# Apply the correction function
data[['First Name', 'Last Name']] = data.apply(correct_name_order, axis=1)

# Save the corrected dataset
corrected_file_path = '/content/drive/My Drive/IEA/iea_output_corrected.csv'
data.to_csv(corrected_file_path, index=False)

print(f"Corrected dataset saved to: {corrected_file_path}")


Corrected dataset saved to: /content/drive/My Drive/IEA/iea_output_corrected.csv
